In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


#Read data from drive

In [2]:
import pandas as pd

def stream_csv_1():
    path = "/content/drive/MyDrive/Colab Notebooks/LegalAIDatasets/1. textdata/1.2df_txt_norm.csv"

    for df in pd.read_csv(path, chunksize=1000):
        for text in df['text'].dropna():
            yield str(text)

def stream_csv_2():
    path = "/content/drive/MyDrive/Colab Notebooks/LegalAIDatasets/1. textdata/2.2df_CSV_norm.csv"

    for df in pd.read_csv(path, chunksize=1000):
        for text in df['text'].dropna():
            yield str(text)



import pyarrow.parquet as pq
import os

import pyarrow.parquet as pq
import os

def stream_parquet_folder():
    folder = "/content/drive/MyDrive/Colab Notebooks/LegalAIDatasets/1. textdata/3.2_clean_chunks"

    for file in sorted(os.listdir(folder)):
        if file.endswith(".parquet"):
            path = os.path.join(folder, file)
            print(f"📂 Processing: {file}")

            pf = pq.ParquetFile(path)

            for batch in pf.iter_batches(batch_size=1000):
                df = batch.to_pandas()

                for text in df['text'].dropna():
                    yield str(text)

Chunk Function

In [3]:
def chunk_function(text, chunk_size=500, overlap=50):
    text = str(text)
    start = 0

    while start < len(text):
        chunk = text[start:start + chunk_size]

        if len(chunk.strip()) > 50:
            yield chunk

        start += chunk_size - overlap

# Load embedding model

In [ ]:
from sentence_transformers import SentenceTransformer
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

model = SentenceTransformer("all-MiniLM-L6-v2", device=device)
model.max_seq_length = 512

# Optional speed boost (safe to comment if issues)
try:
    model = torch.compile(model)
except:
    pass

print("Using device:", device)

Batch Embedding Function

This takes chunks (list of text) and returns embeddings.

In [5]:
EMBED_BATCH_SIZE = 64

def embed_batch(text_batch):
    embeddings = model.encode(
        text_batch,
        batch_size=EMBED_BATCH_SIZE,
        show_progress_bar=False
    )
    return embeddings

# Streaming Embedding Generator

In [6]:
def generate_embeddings(text_generator, chunk_function, batch_size=64):
    batch_chunks = []
    metadata = []
    seen = set()

    for doc_id, text in enumerate(text_generator):
        for chunk_id, chunk in enumerate(chunk_function(text)):

            # Deduplication
            if chunk in seen:
                continue
            seen.add(chunk)

            batch_chunks.append(chunk)
            metadata.append({
                "doc_id": doc_id,
                "chunk_id": chunk_id
            })

            if len(batch_chunks) >= batch_size:
                embeddings = embed_batch(batch_chunks)

                yield batch_chunks, embeddings, metadata

                batch_chunks = []
                metadata = []

    # Final batch
    if batch_chunks:
        embeddings = embed_batch(batch_chunks)
        yield batch_chunks, embeddings, metadata

In [9]:
generate_embeddings

<function __main__.generate_embeddings(text_generator, chunk_function, batch_size=64)>

In [8]:
for chunks, embeddings, metadata in generate_embeddings(stream_csv_1(), chunk_function):

    vectors = []

    for chunk, emb, meta in zip(chunks, embeddings, metadata):
        vectors.append({
            "id": f"{meta['doc_id']}_{meta['chunk_id']}",
            "values": emb.tolist(),
            "metadata": {
                "text": chunk
            }
        })

    print(f"Processed batch: {len(vectors)} vectors")

    # (uncomment when ready)
    # index.upsert(vectors=vectors)

Streaming output truncated to the last 5000 lines.
Processed batch: 64 vectors
Processed batch: 64 vectors
Processed batch: 64 vectors
Processed batch: 64 vectors
Processed batch: 64 vectors
Processed batch: 64 vectors
Processed batch: 64 vectors
Processed batch: 64 vectors
Processed batch: 64 vectors
Processed batch: 64 vectors
Processed batch: 64 vectors
Processed batch: 64 vectors
Processed batch: 64 vectors
Processed batch: 64 vectors
Processed batch: 64 vectors
Processed batch: 64 vectors
Processed batch: 64 vectors
Processed batch: 64 vectors
Processed batch: 64 vectors
Processed batch: 64 vectors
Processed batch: 64 vectors
Processed batch: 64 vectors
Processed batch: 64 vectors
Processed batch: 64 vectors
Processed batch: 64 vectors
Processed batch: 64 vectors
Processed batch: 64 vectors
Processed batch: 64 vectors
Processed batch: 64 vectors
Processed batch: 64 vectors
Processed batch: 64 vectors
Processed batch: 64 vectors
Processed batch: 64 vectors
Processed batch: 64 vecto